# تست نسخه‌ی ۳: استفاده‌ی مستقیم از s2u_map (که در تست v2 تأیید شد درست کار می‌کند)

## نتیجه‌ی تست v2
خروجی v2 نشان داد:
- `s2u_map` و `u2s_map` **کاملاً درست** کار می‌کنند: هر ۹ اتم متوالی در سوپرسل (`[0]*9 + [9]*9 + [18]*9 + ...`)
  به همان اتم سلول واحد نگاشت می‌شوند. یعنی فرمت `s2u_map[i]` = اندیس **در آرایه‌ی سوپرسل**
  نماینده‌ی آن گروه (نه اندیس در سلول واحد به‌تنهایی)
- مشکل واقعی در تست v2 جای دیگری بود: حدس `dim_guess=(1,1,9)` و استفاده از `unitcell.cell`
  (به‌جای `supercell.cell` واقعی) برای ساخت بردارهای جستجوی R، باعث شد روش brute-force PBC
  جواب درستی ندهد (۷ گروه نامتوازن به‌جای ۹ گروه مساوی)

## رویکرد این نسخه
دیگر نیازی به brute-force PBC و حدس `dim_guess` نیست. مستقیماً از `s2u_map` که تأیید شد
درست است استفاده می‌کنیم:
1. اتم‌هایی با `s2u_map` یکسان، به یک "گروه" (مرتبط با یک اتم سلول واحد) تعلق دارند
2. بردار جابجایی هر اتم = `position_supercell[i] - position_supercell[representative_idx]`
   که `representative_idx = s2u_map[i]` (همان اتم مرجع گروه که خودش هم در سوپرسل است)
3. این بردار جابجایی، برای اتم‌هایی که به "همان تصویر" تعلق دارند (نه گروه مرجع)، باید
   بین گروه‌های مختلف یکسان باشد — این را تأیید می‌کنیم

In [ ]:
!pip install -q phonopy -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("✅ نصب شد")

In [ ]:
import numpy as np
import yaml
from pathlib import Path
from phonopy import Phonopy
from phonopy.structure.atoms import PhonopyAtoms
from phonopy.file_IO import parse_FORCE_CONSTANTS

FC_DIR     = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
BANDS_DIR  = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'

FC_DIR_PATH = Path(FC_DIR)
BANDS_DIR_PATH = Path(BANDS_DIR)

fc_files = {f.stem: f for f in FC_DIR_PATH.glob('*.fc')}
band_yaml_files = {f.stem: f for f in BANDS_DIR_PATH.glob('*.yaml')}

common = sorted(set(fc_files) & set(band_yaml_files))
test_formula = common[0]
print(f"ماده‌ی تستی: {test_formula}")

with open(band_yaml_files[test_formula]) as f:
    data = yaml.safe_load(f)

lattice = np.array(data['lattice'])
symbols = [p['symbol'] for p in data['points']]
frac_coords = np.array([p['coordinates'] for p in data['points']])
n_atoms = len(symbols)
print(f"n_atoms (سلول واحد): {n_atoms}")

unitcell = PhonopyAtoms(symbols=symbols, cell=lattice, scaled_positions=frac_coords)

fc = parse_FORCE_CONSTANTS(str(fc_files[test_formula]))
n_supercell = fc.shape[1]
n_images = n_supercell // n_atoms
print(f"n_supercell: {n_supercell} -> n_images: {n_images}")

def guess_dim(n):
    for a in range(1, n+1):
        if n % a == 0:
            rem = n // a
            for b in range(1, rem+1):
                if rem % b == 0:
                    c = rem // b
                    return (a, b, c)
    return (n, 1, 1)

dim_guess = guess_dim(n_images)
print(f"ابعاد سوپرسل (حدس -- فقط برای ساخت آبجکت Phonopy لازم است، نتیجه‌ی این تست به آن حساس نیست): {dim_guess}")

phonon = Phonopy(unitcell, supercell_matrix=[[dim_guess[0],0,0],[0,dim_guess[1],0],[0,0,dim_guess[2]]])
phonon.force_constants = fc
print(f"✅ Phonopy ساخته شد. n_atoms سوپرسل: {len(phonon.supercell.symbols)}")

## استفاده‌ی مستقیم از s2u_map (بدون نیاز به brute-force PBC یا حدس ابعاد)

In [ ]:
supercell = phonon.supercell
sc_cart = supercell.positions  # (n_supercell, 3) کارتزی
s2u_map = supercell.s2u_map    # (n_supercell,) -- اندیس نماینده‌ی هر اتم در همان آرایه‌ی سوپرسل

print(f"s2u_map یکتا (باید n_atoms={n_atoms} مقدار باشد): {np.unique(s2u_map)}")

# بردار جابجایی هر اتم نسبت به نماینده‌ی گروهش
displacement_vs_representative = sc_cart - sc_cart[s2u_map]
print(f"\nشکل displacement_vs_representative: {displacement_vs_representative.shape}")
print(f"چند نمونه‌ی اول:\n{displacement_vs_representative[:12]}")

## گروه‌بندی بر اساس "موقعیت نسبی درون گروه" (image index) به‌جای s2u_map

`s2u_map` می‌گوید هر اتم به کدام **اتم سلول واحد** (که در سوپرسل تکرار شده) متعلق است،
نه اینکه کدام "تصویر سوپرسل" است. برای پیدا کردن image index واقعی هر اتم (بین ۰ تا
n_images-1)، باید ببینیم اتم چندمین عضو گروه خودش است (یعنی jth کپی از آن اتم سلول واحد).

به عبارت دیگر: همه‌ی اتم‌هایی که "image یکسان" دارند، باید بردار جابجایی یکسانی نسبت به
نماینده‌ی گروه خودشان داشته باشند (هرچند گروه‌هایشان متفاوت است).

In [ ]:
# برای هر گروه (بر اساس s2u_map)، لیست اندیس‌های عضو آن گروه را پیدا می‌کنیم،
# و آن‌ها را بر اساس فاصله‌شان از نماینده‌ی گروه مرتب می‌کنیم تا تطابق "تصویر یکسان" بین گروه‌ها روشن شود

unique_reps = np.unique(s2u_map)
print(f"تعداد گروه‌های یکتا (باید n_atoms={n_atoms} باشد): {len(unique_reps)}")

groups = {}
for rep in unique_reps:
    member_idx = np.where(s2u_map == rep)[0]
    groups[rep] = member_idx
    print(f"  نماینده {rep}: اعضا = {member_idx} (تعداد: {len(member_idx)}, انتظار: {n_images})")

# آیا همه‌ی گروه‌ها اندازه‌ی یکسان (n_images) دارند؟
all_same_size = all(len(v) == n_images for v in groups.values())
print(f"\n✅ همه‌ی گروه‌ها اندازه‌ی n_images={n_images} دارند: {all_same_size}")

In [ ]:
# حالا برای هر گروه، بردار جابجایی هر عضو نسبت به نماینده‌ی همان گروه را حساب می‌کنیم
# و آن‌ها را بر اساس بزرگی (norm) مرتب می‌کنیم تا ترتیب "تصویر" در هر گروه قابل‌مقایسه با گروه‌های دیگر شود

per_group_displacements = {}
for rep, members in groups.items():
    disp = sc_cart[members] - sc_cart[rep]  # (n_images, 3)
    per_group_displacements[rep] = disp

# مرتب‌سازی اعضای هر گروه بر اساس norm بردار جابجایی (ترتیب ثابت و قابل‌تکرار)
sorted_displacements_per_group = []
for rep in unique_reps:
    disp = per_group_displacements[rep]
    norms = np.linalg.norm(disp, axis=1)
    order = np.argsort(norms)
    sorted_displacements_per_group.append(disp[order])

sorted_displacements_per_group = np.array(sorted_displacements_per_group)  # (n_atoms, n_images, 3)
print(f"شکل sorted_displacements_per_group: {sorted_displacements_per_group.shape}")

# بررسی: آیا بردار جابجایی در image index ثابت، بین گروه‌های مختلف یکسان است؟
# یعنی sorted_displacements_per_group[:, k, :] باید برای همه‌ی گروه‌ها (تقریبا) یکسان باشد
print(f"\nبرای هر image index k، انحراف استاندارد بین گروه‌ها (باید نزدیک صفر باشد):")
for k in range(n_images):
    vals_at_k = sorted_displacements_per_group[:, k, :]  # (n_atoms, 3)
    std_at_k = np.std(vals_at_k, axis=0)
    mean_at_k = np.mean(vals_at_k, axis=0)
    print(f"  image {k}: mean_disp={mean_at_k}, std_across_groups={std_at_k}")

## نتیجه‌گیری نهایی

اگر در سلول بالا، برای هر `image k`، مقدار `std_across_groups` نزدیک صفر باشد (مثلاً
کمتر از `1e-3`)، یعنی:
- بردار جابجایی هر "تصویر" (بعد از مرتب‌سازی بر اساس norm درون هر گروه) بین تمام ۸ اتم
  سلول واحد یکسان است
- یعنی توانستیم یک بردار جابجایی واقعی و معنادار `(n_images, 3)` برای کل ماده استخراج کنیم
  که در همه‌ی گروه‌ها یکی است — دقیقاً همان چیزی که برای جایگزینی `nn.Embedding` در
  Notebook 17 لازم داریم

⚠️ اگر مقدار std بزرگ بود، یعنی فرض "مرتب‌سازی بر اساس norm، تصاویر متناظر بین گروه‌ها را
هم‌تراز می‌کند" درست نیست و باید روش دیگری (مثلاً مرتب‌سازی بر اساس بردار کامل، نه فقط
norm) را امتحان کنیم.

**لطفاً کل خروجی این نوت‌بوک، به‌خصوص جدول آخر (`image k: mean_disp=...، std_across_groups=...`)
را برای من بفرستید.**